# Training Pipeline — RegNet (pycls)

The same pipeline as [`resnet50.ipynb`](resnet50.ipynb) and the
[template](../../../_templates/Evaluation_Template.ipynb), but the backbone is a **RegNet built by
[pycls](https://github.com/facebookresearch/pycls)** — Facebook Research's reference
implementation from *Designing Network Design Spaces*. Data, loss, optimizer, schedule, training
loop and metrics are untouched, so the runs stay comparable across notebooks.

**Why pycls needs a wrapper.** Everywhere else the pipeline calls `Model(num_classes=...)`. pycls
does not work that way: `builders.build_model()` reads a **global yacs config**, which
`model_zoo.build_model(name)` fills in by downloading that model's YAML (`REGNET.WA`, `W0`, `WM`,
`DEPTH`, `GROUP_W`, …) from the repo. [`_handlers/pycls_models.py`](../../../_handlers/pycls_models.py)
wraps that into the `num_classes -> Module` callable the pipeline's `model_classes` map expects, so
[`evaluation.py`](../../../_handlers/evaluation.py) needs no changes at all.

Two repos get cloned: **this** one (for the `_handlers` package) and **pycls** (for the models).

Use a GPU (a Colab **T4** is enough) — `build_model` calls `.cuda()` on the network.

## Install Requirements

The pipeline's own dependencies, then pycls'. Note pycls' `setup.py` under-declares: importing
`pycls.models` pulls in `pycls.core.io` (needs `iopath`) and `pycls.core.distributed` (needs
`submitit`), neither of which is in `install_requires`. Installing the four packages by hand and
putting the clone on `sys.path` is more predictable here than `pip install -e .`, which would
re-resolve `numpy` against the torch wheel installed just above.

In [1]:
!nvidia-smi

Mon Jul 27 16:17:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 --index-url https://download.pytorch.org/whl/cu124

In [ ]:
!pip install timm medmnist==3.0.2 scikit-learn tqdm requests

In [ ]:
!pip install yacs iopath submitit simplejson

## Get the survey code

The pipeline lives in this repository's `survey/src/_handlers` package, so the notebook needs a
checkout of it. On Colab it clones into `/content/quantum-quantization`, or `git pull --ff-only`s
that directory if it is already there — so re-running the cell after a push picks up the new code.
Run locally, the notebook already sits inside the repo, so `find_src` climbs to `survey/src` and
git is never touched (your working tree is left alone).

The branch is chosen by *environment*, not by working directory: the imports cell `os.chdir`s into
the checkout, so a cwd-based test would find `_handlers` on every re-run and silently skip the pull.

If the repository is private the anonymous clone fails with an authentication error; use a token
URL instead — `REPO_URL = 'https://<GITHUB_TOKEN>@github.com/alexandrachirita98/quantum-quantization.git'`.

In [5]:
import pathlib
import subprocess
import sys

REPO_URL = 'https://github.com/alexandrachirita98/quantum-quantization.git'

# NB: key off the environment, not the working directory. `os.chdir` in the imports cell moves the
# cwd *inside* the checkout, so a cwd-based test would report "already have the code" on every
# re-run and silently skip the pull.
IN_COLAB = 'google.colab' in sys.modules or pathlib.Path('/content').is_dir()
CLONE_DIR = pathlib.Path('/content/quantum-quantization')   # where the Colab checkout goes


def find_src(start):
    """Climb from `start` looking for the survey `src/` root — the directory holding `_handlers`."""
    for p in [pathlib.Path(start), *pathlib.Path(start).parents]:
        if (p / '_handlers').is_dir():
            return p
    return None


if IN_COLAB:
    if CLONE_DIR.exists():                                  # refresh whatever was cloned earlier
        subprocess.run(['git', 'pull', '--ff-only'], cwd=str(CLONE_DIR), check=True)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(CLONE_DIR)], check=True)
    SRC = CLONE_DIR / 'survey' / 'src'
else:                                                       # local: the notebook lives in the repo
    SRC = find_src(pathlib.Path.cwd())

assert SRC is not None and (SRC / '_handlers').is_dir(), f'no _handlers package under {SRC}'
print('survey src:', SRC)

survey src: /content/quantum-quantization/survey/src


## Get pycls

pycls is not on PyPI, so it has to come from git. The clone lands next to the survey checkout and
goes on `sys.path` — `PYCLS` is the directory *containing* the `pycls` package, which is what makes
`import pycls` resolve. Re-running the cell does not re-clone.

In [6]:
PYCLS_URL = 'https://github.com/facebookresearch/pycls.git'

# a fixed location, again not cwd-relative: /content on Colab, a cache dir locally so the clone
# never lands inside this repo
PYCLS = pathlib.Path('/content/pycls') if IN_COLAB else pathlib.Path.home() / '.cache' / 'pycls'
if not PYCLS.exists():
    PYCLS.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', PYCLS_URL, str(PYCLS)], check=True)

assert (PYCLS / 'pycls' / 'models' / 'regnet.py').is_file(), f'pycls package not found under {PYCLS}'
print('pycls:', PYCLS)

pycls: /content/pycls


## Imports

`INFO` comes straight from the `medmnist` package (it carries each dataset's `task` and label
map); `build_dataset`, `pycls_factory` and the training/metric helpers come from the survey's
`_handlers`.

Both clones go on `sys.path`, and `os.chdir` moves into `SRC` so the `./data` folder the pipeline
reads and writes is the shared [`src/data`](../../../data) — the same path the medmnist `Evaluator`
uses inside `evaluation.py`.

In [7]:
import os
import sys

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
from medmnist import INFO

# import the pipeline from the survey `src/` root, and work from there so './data' resolves inside it
sys.path.insert(0, str(SRC))
sys.path.insert(0, str(PYCLS))
os.chdir(SRC)
print('working directory:', os.getcwd())

# drop cached handler modules, so a `git pull` above is actually reflected on a re-run
for _m in [m for m in list(sys.modules) if m == '_handlers' or m.startswith('_handlers.')]:
    del sys.modules[_m]

from _handlers.datasets import build_dataset
from _handlers.pycls_models import pycls_factory
from _handlers.evaluation import (
    build_model,
    train_mnist,
    evaluate_all_metrics,
)

working directory: /content/quantum-quantization/survey/src


## Configuration

`main.py` reads its settings from `argparse` on the command line. Here we replace that block with a
plain config object, so the exact same `args` flows through the pipeline.

**Model** — a pycls model-zoo name. RegNetY (with squeeze-excite) and RegNetX come in
`200MF, 400MF, 600MF, 800MF, 1.6GF, 3.2GF, 4.0GF, 6.4GF, 8.0GF, 12GF, 16GF, 32GF`; the zoo also
holds `ResNet-50/101/152`, `ResNeXt-50/101/152` and `EfficientNet-B0..B5`. `RegNetY-800MF` is the
rough compute match for a ResNet-50 at this input size. With `pretrained=True` pycls downloads its
ImageNet checkpoint and the factory swaps the 1000-way head for an `nb_classes` one; left at
`False` to match the other notebooks' train-from-scratch setting.

**Dataset** — any MedMNIST flag: `tissuemnist, pathmnist, chestmnist, dermamnist, octmnist,
pneumoniamnist, retinamnist, breastmnist, bloodmnist, organamnist, organcmnist, organsmnist`. The
first download can take a while.

In [9]:
from types import SimpleNamespace

args = SimpleNamespace(
    model_name='RegNetY-800MF',                      # any pycls model-zoo name
    dataset='breastmnist',                           # a medmnist flag
    batch_size=24,
    lr=1e-4,
    epochs=10,
    pretrained=False,                                # load pycls' ImageNet weights
    checkpoint_path=None,                            # unused on the pycls path
)

## Device

In [10]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using {} device.".format(device))

Using cuda:0 device.


## Dataset

The `task` field from `INFO` (multi-label vs. multi-class) selects the loss. `build_dataset` then
downloads the `.npz` into `./data`, serves it as 3-channel 224×224, applies the transforms and
returns the datasets plus the number of classes.

The transforms are MedViTV2's (normalize at mean/std `0.5`), kept identical across the survey's
notebooks. pycls' own ImageNet recipe normalizes differently — irrelevant when training from
scratch, worth knowing if you flip `pretrained=True`.

In [11]:
# the medmnist `task` selects the loss
info = INFO[args.dataset]
task = info['task']
if task == "multi-label, binary-class":
    loss_function = nn.BCEWithLogitsLoss()
else:
    loss_function = nn.CrossEntropyLoss()

train_dataset, test_dataset, nb_classes = build_dataset(args=args)

print(train_dataset)
print("===================")
print(test_dataset)

Number of channels:  1
Number of classes:  2


100%|██████████| 30.9M/30.9M [00:41<00:00, 745kB/s]


Using downloaded and verified file: ./data/breastmnist_224.npz
Dataset BreastMNIST of size 224 (breastmnist_224)
    Number of datapoints: 546
    Root location: ./data
    Split: train
    Task: binary-class
    Number of channels: 1
    Meaning of labels: {'0': 'malignant', '1': 'normal, benign'}
    Number of samples: {'train': 546, 'val': 78, 'test': 156}
    Description: The BreastMNIST is based on a dataset of 780 breast ultrasound images. It is categorized into 3 classes: normal, benign, and malignant. As we use low-resolution images, we simplify the task into binary classification by combining normal and benign as positive and classifying them against malignant as negative. We split the source dataset with a ratio of 7:1:2 into training, validation and test set. The source images of 1×500×500 are resized into 1×28×28.
    License: CC BY 4.0
Dataset BreastMNIST of size 224 (breastmnist_224)
    Number of datapoints: 156
    Root location: ./data
    Split: test
    Task: binary-

## Model

`build_model` looks the name up in `model_classes` and calls the hit with `num_classes`, then moves
the result to CUDA — so registering the pycls factory under the model's own name is all it takes.
The factory downloads the model's YAML into pycls' cache (`/tmp/pycls-download-cache`), resets the
global config, merges `MODEL.NUM_CLASSES`, and builds.

`pretrained=False` in the `build_model` call is **not** the same switch as `args.pretrained`: it
only tells `evaluation.build_model` to skip the MedViT checkpoint branch (which knows nothing about
pycls names). pycls' own ImageNet weights are handled inside the factory, from `args.pretrained`.

In [12]:
model_classes = {args.model_name: pycls_factory(args.model_name, pretrained=args.pretrained)}

net = build_model(args.model_name, nb_classes, model_classes, pretrained=False)
print(net.head)

  [============================================================] 100.0% of 0.0MB file  
AnyHead(
  (avg_pool): AdaptiveAvgPool2d(output_size=(1, 1))
  (fc): Linear(in_features=768, out_features=2, bias=True)
)


## Optimizer, Scheduler & Data Loaders

AdamW with weight decay and a cosine-annealing schedule stepped **every iteration** — so `T_max` is
the total number of optimizer steps (`epochs * train_num // batch_size`).

pycls' own recipe for these models is SGD + momentum at a much larger batch size; this notebook
deliberately keeps the template's optimizer so the backbone is the only thing that changed.

In [13]:
train_num = len(train_dataset)
eta = args.epochs * train_num // args.batch_size   # total scheduler steps

optimizer = optim.AdamW(net.parameters(), lr=args.lr, betas=[0.9, 0.999], weight_decay=0.05)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=eta, eta_min=5e-6)

train_loader = data.DataLoader(dataset=train_dataset, batch_size=args.batch_size, shuffle=True)
test_loader = data.DataLoader(dataset=test_dataset, batch_size=2*args.batch_size, shuffle=False)

## Train

`train_mnist` (from the handler) runs the loop, scoring every epoch with the medmnist `Evaluator`
(AUC/ACC) and writing the best model to `save_path`.

In [14]:
save_path = f'./{args.model_name}_{args.dataset}.pth'

train_mnist(args.epochs, net, train_loader, test_loader,
            optimizer, scheduler, loss_function, device, save_path, args.dataset, task)

100%|██████████| 4/4 [00:00<00:00,  8.12it/s]
[epoch 1] train_loss: 0.582  auc: 0.571  acc: 0.731

Saving checkpoint...
100%|██████████| 4/4 [00:00<00:00, 11.87it/s]
[epoch 2] train_loss: 0.558  auc: 0.667  acc: 0.712

Saving checkpoint...
100%|██████████| 4/4 [00:00<00:00, 11.81it/s]
[epoch 3] train_loss: 0.549  auc: 0.715  acc: 0.756

Saving checkpoint...
100%|██████████| 4/4 [00:00<00:00, 11.71it/s]
[epoch 4] train_loss: 0.550  auc: 0.722  acc: 0.744

Saving checkpoint...
100%|██████████| 4/4 [00:00<00:00,  9.86it/s]
[epoch 5] train_loss: 0.561  auc: 0.655  acc: 0.750
100%|██████████| 4/4 [00:00<00:00, 11.33it/s]
[epoch 6] train_loss: 0.556  auc: 0.671  acc: 0.744
100%|██████████| 4/4 [00:00<00:00, 11.68it/s]
[epoch 7] train_loss: 0.562  auc: 0.707  acc: 0.731
100%|██████████| 4/4 [00:00<00:00, 11.28it/s]
[epoch 8] train_loss: 0.548  auc: 0.722  acc: 0.731

Saving checkpoint...
100%|██████████| 4/4 [00:00<00:00,  9.74it/s]
[epoch 9] train_loss: 0.531  auc: 0.704  acc: 0.737
100%|███

## Evaluate all metrics

`evaluate_all_metrics` (from the handler) runs the model once over one split and prints **every**
metric the training routines can produce: the medmnist Evaluator AUC/ACC plus accuracy, weighted
precision / recall (sensitivity) / F1, per-class + average specificity, one-vs-rest AUC, the
confusion matrix and a per-class report. Set `split` to `'train'` or `'test'`; uncomment the
`load_state_dict` line to score the best checkpoint instead of the in-memory model.

In [15]:
split = 'test'   # 'train' or 'test'

# net.load_state_dict(torch.load(save_path)['model'])   # uncomment to evaluate the BEST checkpoint

eval_dataset = train_dataset if split == 'train' else test_dataset
metrics = evaluate_all_metrics(net, eval_dataset, args.dataset, nb_classes, device,
                               split=split, batch_size=2 * args.batch_size)

100%|██████████| 4/4 [00:00<00:00,  9.47it/s]
[medmnist Evaluator]  auc: 0.6529  acc: 0.7436

=== test metrics (2 classes) ===
accuracy            : 0.7436
overall_accuracy    : 0.7436
auc (ovr)           : 0.6529
precision (weighted): 0.8102
recall / sensitivity: 0.7436
specificity (avg)   : 0.5238
f1 (weighted)       : 0.6462

per-class specificity: ['1.000', '0.048']

confusion matrix:
[[  2  40]
 [  0 114]]

classification report:
              precision    recall  f1-score   support

           0       1.00      0.05      0.09        42
           1       0.74      1.00      0.85       114

    accuracy                           0.74       156
   macro avg       0.87      0.52      0.47       156
weighted avg       0.81      0.74      0.65       156

